Cell 1: Install System and Python DependenciesThis block configures the Linux backend, installs the necessary optical character recognition (OCR) engines, and installs Marker.

In [ ]:
# Install required system packages for PDF processing and OCR
!apt-get update && apt-get install -y poppler-utils tesseract-ocr libgl1 libglx-mesa0 libglib2.0-0 zip

# Upgrade pip and install Marker
!pip install --upgrade pip
!pip install marker-pdf pypdf


Cell 2: Upload Your TextbookRun this block to open an upload prompt. Click "Choose Files" and upload your scanned textbook.Note: Once uploaded, right-click your PDF file in the left sidebar, select Copy path, and make sure it matches the path in Cell 3.

In [ ]:
from google.colab import files
import os

print("Upload your textbook PDF file:")
uploaded = files.upload()

# Print the name of the file you uploaded to confirm
for filename in uploaded.keys():
    print(f"Successfully uploaded: {filename} (Path: /content/{filename})")


Cell 3: Configure PathsDefine your input book path and target output directory. (Adjust textbook.pdf if your file has a different name).

In [ ]:
# Define paths for processing
BOOK_PATH = "/content/textbook.pdf"  # Update this if your file name is different
OUTPUT_DIR = "/content/final_output"

import os
if not os.path.exists(BOOK_PATH):
    print(f"⚠️ Warning: Could not find a file at {BOOK_PATH}. Please check your filename in the left sidebar.")
else:
    print(f"✅ Target file verified at {BOOK_PATH}")


Cell 4: Run Marker at Full ScaleBecause Google Colab provides a dedicated Nvidia GPU with native VRAM, we can remove all chunking workarounds. Marker will process the entire textbook in a single parallel run.

In [ ]:
import subprocess
import os

if os.path.exists(BOOK_PATH):
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Execute marker natively using the cloud GPU
    command = [
        "marker_single",
        BOOK_PATH,
        "--output_dir", OUTPUT_DIR
    ]

    print(f"Starting GPU accelerated conversion for {BOOK_PATH}...")
    print("This will process the entire book without chunking. Please wait...\n")

    # Run the command and stream output directly to the Colab console
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

    for line in process.stdout:
        print(line, end="")

    process.wait()

    if process.returncode == 0:
        print("\n🎉 Conversion completed successfully!")
    else:
        print(f"\n❌ Process failed with exit code {process.returncode}")


Cell 5: Zip and Download the ResultsThis final block compresses your generated Markdown file and your folder of graph images into a single zip archive and downloads it directly to your local computer.

In [ ]:
import os
from google.colab import files

if os.path.exists(OUTPUT_DIR):
    print("Compressing output markdown and graph figures...")
    # Zip the final_output directory
    !zip -q -r /content/processed_textbook.zip {OUTPUT_DIR}

    print("Downloading zip archive to your machine...")
    files.download('/content/processed_textbook.zip')
else:
    print("Error: Output directory not found. Did Cell 4 complete successfully?")
